# LLM Basics: Working with Different LLM Providers

This notebook demonstrates how to interact with various Large Language Model (LLM) providers including OpenAI, Google Gemini, and Groq.

## Learning Objectives
- Understand how to authenticate with different LLM providers
- Send prompts and receive responses from LLMs
- Compare response formats across providers
- Apply structured prompting techniques

## Related Theoretical Content
- [AI Fundamentals](../../notes/03-ai/README.md)
- [Generative AI Concepts](../../notes/03-ai/01-fundamentals/README.md)
- [Prompt Engineering](../../notes/03-ai/02-prompt_engineering/README.md)

## Prerequisites
- Run `setup.ps1` (Windows) or `setup.sh` (Mac/Linux) to create virtual environment
- Configure your API keys in `api_keys.py`

## Setup: Import API Keys

Import our centralized API key management system. This keeps credentials secure and separated from code.

In [ ]:
# Import the API key management module
try:
    from api_keys import get_api_key, validate_keys

    # Validate that keys are configured
    validate_keys()
except ImportError:
    print(" api_keys.py not found!")
    print("   Please run setup.ps1 or setup.sh first.")
except Exception as e:
    print(f"  {e}")
    print("   Some providers may not work without proper API keys.")

## 1. OpenAI GPT-4o-mini

OpenAI provides state-of-the-art language models. The GPT-4o-mini model offers a good balance of capability and cost.

### Key Concepts:
- **Client initialization**: Creates a connection to OpenAI's API
- **Chat completions**: Conversation-style API where we provide messages with roles (system, user, assistant)
- **System message**: Sets the behavior/context for the AI
- **User message**: The actual query or prompt

In [ ]:
from openai import OpenAI

# Initialize the OpenAI client with our API key
client = OpenAI(api_key=get_api_key('openai'))

### Basic Data Extraction Example

This example demonstrates using a system prompt to define the AI's role and a user message to provide input. The model extracts structured information (name, age, location) from unstructured text.

In [ ]:
# Create a chat completion request
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "You are a data extraction assistant. You will be given some user input which includes personal information. Extract the person's name, age and location from the input and return it in a JSON format.",
        },
        {
            "role": "user",
            "content": "Hi, this is John Doe. I am 30 years old and I live in New York.",
        }
    ]
)

# Display the response
print("Extracted Data:")
print(response.choices[0].message.content)

# Display usage statistics
print(f"\nTokens used: {response.usage.total_tokens}")

## 2. Google Gemini

Google's Gemini is a multimodal AI model that can understand and generate text, code, images, and more.

### Key Differences from OpenAI:
- Uses `generate_content` instead of chat completions
- System instructions are passed via `GenerateContentConfig`
- Temperature and token limits configured differently

In [ ]:
from google import genai
from google.genai import types

# Initialize the Gemini client
client = genai.Client(api_key=get_api_key('google'))

### Same Data Extraction with Gemini

This demonstrates the same extraction task using Google's Gemini model, showing how different providers handle the same prompt.

In [ ]:
# Generate content with Gemini
response = client.models.generate_content(
    model="gemini-2.0-flash-exp",
    contents="My name is John Doe. I am 30 years old and I live in New York.",
    config=types.GenerateContentConfig(
        system_instruction="You are a data extraction assistant. You will be given some user input which includes personal information. Extract the person's name, age and location from the input and return it in a JSON format.",
        temperature=0.6,
        max_output_tokens=2048
    ),
)

print("Extracted Data (Gemini):")
print(response.text)

## 3. Groq (Fast LLM Inference)

Groq provides extremely fast inference using specialized hardware. They host open-source models like Llama with impressive speed.

### Key Features:
- Very fast response times
- Uses familiar OpenAI-compatible API
- Supports various open-source models (Llama, Mixtral, etc.)

In [ ]:
from groq import Groq

# Initialize Groq client
client = Groq(api_key=get_api_key('groq'))

### Basic Extraction with Groq/Llama

Using the Llama 3.3 70B model via Groq's fast inference platform. Note the similar API structure to OpenAI.

In [ ]:
# Create chat completion
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You are a data extraction assistant. You will be given some user input which includes personal information. Extract the person's name, age and location from the input and return it in a JSON format.",
        },
        {
            "role": "user",
            "content": "Hi, this is John Doe. I am 30 years old and I live in New York.",
        }
    ],
    model="llama-3.3-70b-versatile",
    temperature=0.5,
    max_completion_tokens=1024,
)

print("Extracted Data (Groq/Llama):")
print(chat_completion.choices[0].message.content)

## 4. Advanced Example: Medical Record Extraction

A more complex extraction task with multiple entities and specific output requirements.

### Prompt Engineering Techniques:
- **Role definition**: Clearly state the AI's purpose
- **Structured instructions**: Define expected entities and format
- **Constraints**: Specify handling of missing data
- **Schema specification**: Provide exact JSON structure

In [ ]:
# Re-initialize Groq client for this example
client = Groq(api_key=get_api_key('groq'))

# Define a detailed system prompt with clear structure
system_prompt = '''
##ROLE##
You are a patient data extraction assistant.

##INSTRUCTIONS##
Refer user input extract these entities ["Name", "Age", "Diagnosis", "Medication"] and return it in a JSON format specified.

JSON Schema:
{
    "Name": string,
    "Age": integer,
    "Diagnosis": string,
    "Medication": string
}

##CONSTRAINTS##
Output must only be a JSON object that adheres to the above schema. Do not include any additional text or explanations. If an entity is not present in the user input, set its value to null in the JSON output.
The input must contain at lease one of the entities. If none of the entities are present, return a JSON object with all values set to null.
'''

user_input = "Patient name is Jane Smith, she is 45 years old and has been diagnosed with hypertension. She is currently taking Lisinopril."

chat_completion = client.chat.completions.create(
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ],
    model="llama-3.3-70b-versatile",
    temperature=0,  # Use 0 for deterministic extraction
    max_completion_tokens=1024,
)

print("Medical Record Extraction:")
print(chat_completion.choices[0].message.content)

## 5. Customer Support Analysis

This example demonstrates a more complex use case: analyzing customer support conversations to extract multiple insights.

### What This Does:
1. Loads a conversation transcript from a file
2. Analyzes sentiment, issues, and resolution quality
3. Scores agent performance and policy compliance
4. Generates a structured summary

### Prepare Sample Transcript

Create a sample transcript file if one doesn't exist.

In [ ]:
import os

# Create sample transcript if it doesn't exist
transcript_path = '.data/transcript.txt'
os.makedirs('.data', exist_ok=True)

if not os.path.exists(transcript_path):
    sample_transcript = """Agent Sarah: Hello, thank you for contacting support. My name is Sarah. How can I help you today?

Customer Mike Johnson: Hi Sarah, I'm having trouble logging into my account. It keeps saying my password is incorrect.

Agent Sarah: I'm sorry to hear that, Mike. Let me help you with that. Can you confirm the email address associated with your account?

Customer Mike Johnson: Yes, it's mike.johnson@email.com

Agent Sarah: Thank you. I can see your account. Let me send you a password reset link to that email. You should receive it within 2 minutes.

Customer Mike Johnson: Great, I got it! I was able to reset my password and log in successfully.

Agent Sarah: Wonderful! Is there anything else I can help you with today?

Customer Mike Johnson: No, that's all. Thank you for your quick help!

Agent Sarah: You're welcome! Have a great day!"""

    with open(transcript_path, 'w') as f:
        f.write(sample_transcript)
    print(" Created sample transcript")
else:
    print("Transcript already exists")

### Analyze the Conversation

Use an LLM to extract structured insights from the unstructured conversation.

In [ ]:
client = Groq(api_key=get_api_key('groq'))

system_prompt = '''
##ROLE##
You are a customer support conversations analyzer.
You will analyze past customer support conversations and extract key information such as customer sentiment,
main issues, and resolution status.
You will also evaluate performance, tone and professionalism of the support agents based on the conversations.

##INSTRUCTIONS##
You will assign a score of 1-5 for all the integer fields based on the conversation, where 1 is the lowest and 5 is the highest.
You will validate the resolution provided with standard corporate policies and best practices.
The output must be generated with the following headers

Agent Name: {string}
Customer Name: {string}
Customer Sentiment: {integer}
Main Issues: {string}
Resolution Status: {integer}
Support Agent Score: {integer}
Policy Compliance Score: {integer}
Summary of Conversation (250 words max): {string}

##CONSTRAINTS##
Output must adhere to the format specified above. Do not include any additional text or explanations. If an entity is not present in the user input, set its value to null in the output.
If the conversation cannot produce any of these fields [Agent Name, Customer Name, Customer Sentiment, Main Issues] fields, simply return "Invalid/Incomplete Conversation".
'''

# Load the transcript
user_input = ''
try:
    with open(transcript_path, 'r') as file:
        user_input = file.read()
except FileNotFoundError:
    print("Error: The transcript file does not exist.")
    user_input = None

if user_input:
    chat_completion = client.chat.completions.create(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ],
        model="llama-3.3-70b-versatile",
        temperature=0,
        max_completion_tokens=1024,
    )

    print("Customer Support Analysis:")
    print(chat_completion.choices[0].message.content)

## Key Takeaways

1. **Provider Differences**: While APIs differ, core concepts (system prompts, user messages, temperature) are universal
2. **Prompt Engineering**: Clear structure and constraints improve output quality
3. **Temperature Control**: Use 0 for deterministic tasks, higher values for creative tasks
4. **Security**: Always use environment variables or secure key management (never hardcode keys)

## Next Steps

- [02-structured-output.ipynb](02-structured-output.ipynb): Learn to enforce JSON schemas with Pydantic
- [03-rag-basics.ipynb](03-rag-basics.ipynb): Build retrieval-augmented generation systems